In [33]:
!pip install dash plotly networkx scikit-learn pandas numpy
!pip install plotly[clouds]

In [34]:
import pandas as pd
import numpy as np
import networkx as nx
from dash import Dash, dcc, html, Input, Output
import plotly.graph_objects as go
import plotly.express as px
from sklearn.decomposition import PCA
from networkx.algorithms import community

In [35]:

# LOAD DATASET

temperature = pd.read_csv("temperature.csv").ffill()
humidity = pd.read_csv("humidity.csv").ffill()
pressure = pd.read_csv("pressure.csv").ffill()
wind = pd.read_csv("wind_speed.csv").ffill()

cities = temperature.columns[1:]

# AGGREGATE METRICS


records = []
for city in cities:
    records.append({
        "city": city,
        "temperature": temperature[city].mean(),
        "humidity": humidity[city].mean(),
        "pressure": pressure[city].mean(),
        "wind": wind[city].mean()
    })

city_df = pd.DataFrame(records)

# NETWORK CONSTRUCTION

corr = temperature[cities].corr()
G = nx.Graph()

for c in cities:
    G.add_node(c)

for i in range(len(cities)):
    for j in range(i+1,len(cities)):
        if corr.iloc[i,j] > 0.7:
            G.add_edge(cities[i],cities[j],weight=corr.iloc[i,j])

# CENTRALITY

city_df["degree"] = city_df.city.map(nx.degree_centrality(G))
city_df["betweenness"] = city_df.city.map(nx.betweenness_centrality(G))

# COMMUNITY DETECTION

comms = community.greedy_modularity_communities(G)

community_map = {}
for i,com in enumerate(comms):
    for node in com:
        community_map[node]=i

city_df["community"] = city_df.city.map(community_map)


# PCA

pca = PCA(n_components=2)
city_df[["PC1","PC2"]] = pca.fit_transform(
    city_df[["temperature","humidity","pressure","wind"]]
)

# DASH APP


app = Dash(__name__)

app.layout = html.Div([

    html.H1("Climate Network Visual Analytics Dashboard"),

    html.Div([
        html.Label("Betweenness Filter"),
        dcc.Slider(
            id="filter",
            min=0,
            max=city_df.betweenness.max(),
            step=0.001,
            value=city_df.betweenness.quantile(0.6)
        )
    ],style={"width":"40%"}),

    html.Div([
        html.Label("Temporal Index"),
        dcc.Slider(
            id="time",
            min=0,
            max=500,
            step=10,
            value=0
        )
    ],style={"width":"40%"}),

    dcc.Graph(id="network"),
    dcc.Graph(id="scatter"),
    dcc.Graph(id="parallel"),
    dcc.Graph(id="pca"),
    dcc.Graph(id="map")
])


# CALLBACK

@app.callback(
    Output("network","figure"),
    Output("scatter","figure"),
    Output("parallel","figure"),
    Output("pca","figure"),
    Output("map","figure"),
    Input("filter","value"),
    Input("time","value"),
    Input("scatter","selectedData")
)
def update(threshold, time_index, selected):
    try:
        # Create empty figure for error cases
        empty = go.Figure().update_layout(title="No Data After Filtering")
        
        # Copy dataframe to avoid modifying original
        df = city_df.copy()
        
        # Apply threshold filter
        df = df[df.betweenness >= threshold]
        
        # Handle selection if available
        if selected and "points" in selected:
            try:
                sel = [p["text"] for p in selected["points"]]
                df = df[df.city.isin(sel)]
            except Exception as e:
                print(f"Selection error: {e}")
        
        # Check if the dataframe is empty after filtering
        if df.empty:
            return empty, empty, empty, empty, empty
        
        # Create subgraph with filtered cities
        try:
            city_list = df.city.tolist()
            Gf = G.subgraph(city_list).copy()
        except Exception as e:
            print(f"Subgraph creation error: {e}")
            return empty, empty, empty, empty, empty
        
        # Check if graph is empty
        if len(Gf.nodes) == 0:
            return empty, empty, empty, empty, empty
        
        # Calculate layout
        try:
            pos = nx.spring_layout(Gf, seed=42)
        except Exception as e:
            print(f"Layout calculation error: {e}")
            return empty, empty, empty, empty, empty
        
        # Add position coordinates to dataframe
        df["x"] = df.city.apply(lambda c: pos[c][0] if c in pos else 0)
        df["y"] = df.city.apply(lambda c: pos[c][1] if c in pos else 0)
        
        # Create network visualization
        try:
            edge_x = []
            edge_y = []
            for e in Gf.edges():
                if e[0] in pos and e[1] in pos:  # Make sure both nodes are in pos
                    x0, y0 = pos[e[0]]
                    x1, y1 = pos[e[1]]
                    edge_x += [x0, x1, None]
                    edge_y += [y0, y1, None]
            
            net = go.Figure()
            net.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(color='#888', width=0.5)))
            net.add_trace(go.Scatter(
                x=df.x, y=df.y, text=df.city,
                mode='markers+text',
                marker=dict(
                    size=df.degree*120,
                    color=df.community,
                    colorscale='Viridis',
                    showscale=True,
                    colorbar=dict(title="Community")
                )
            ))
            net.update_layout(title="Network Visualization", showlegend=False)
        except Exception as e:
            print(f"Network visualization error: {e}")
            net = empty
        
        # Create scatter plot
        try:
            scat = px.scatter(
                df, x="degree", y="betweenness", text="city", color="community",
                title="Degree vs Betweenness Centrality",
                labels={"degree": "Degree Centrality", "betweenness": "Betweenness Centrality"}
            )
        except Exception as e:
            print(f"Scatter plot error: {e}")
            scat = empty
        
        # Create parallel coordinates plot
        try:
            par = px.parallel_coordinates(
                df,
                dimensions=["temperature", "humidity", "pressure", "wind", "degree", "betweenness"],
                color="community",
                title="Parallel Coordinates Plot"
            )
        except Exception as e:
            print(f"Parallel coordinates error: {e}")
            par = empty
        
        # Create PCA plot
        try:
            pca_fig = px.scatter(
                df, x="PC1", y="PC2", text="city", color="community",
                title="PCA Visualization"
            )
        except Exception as e:
            print(f"PCA visualization error: {e}")
            pca_fig = empty
        
        # Create map visualization
        try:
            # Ensure time_index is within bounds
            safe_time_index = min(int(time_index), len(temperature) - 1)
            
            # Get temperature values for the cities in df
            temps = []
            for city in df.city:
                try:
                    if city in temperature.columns:
                        temps.append(temperature.iloc[safe_time_index][city])
                    else:
                        temps.append(0)
                except Exception as e:
                    print(f"Error getting temperature for {city}: {e}")
                    temps.append(0)
            
            # Ensure temps has the same length as df
            if len(temps) != len(df):
                temps = [1] * len(df)  # Default size if there's a mismatch
            
            mapfig = px.scatter_geo(
                df,
                locations="city",
                locationmode="city names",
                size=[abs(t) + 1 for t in temps],  # Add 1 to avoid zero sizes
                color="community",
                title=f"Geographic Distribution (Time Index: {safe_time_index})"
            )
        except Exception as e:
            print(f"Map visualization error: {e}")
            mapfig = empty
        
        return net, scat, par, pca_fig, mapfig
        
    except Exception as e:
        print(f"General callback error: {e}")
        empty = go.Figure().update_layout(title=f"Error: {str(e)}")
        return empty, empty, empty, empty, empty

# RUN

if __name__ == "__main__":
    app.run(debug=True,port=8051)

Author: BONA NASSER